In [1]:
import sklearn
import pandas as pd
import numpy as np
import mlflow
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report
from sklearn.model_selection import train_test_split
from mlflow import log_metric, log_param, log_artifact
from mlflow.sklearn import log_model
from sklearn.datasets import make_classification    
# Load the dataset

In [2]:
X,y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42,weights=[0.7,0.3])
np.unique(y,return_counts=True) 

(array([0, 1]), array([699, 301]))

In [3]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print(f"Train set: {np.unique(y_train,return_counts=True)}")

Train set: (array([0, 1]), array([559, 241]))


In [4]:
params= {'solver':'lbfgs',
         'max_iter':1000,
         'random_state':42,}

lr = LogisticRegression(**params)
lr.fit(X_train,y_train)
y_pred = lr.predict(X_test)
acc = sklearn.metrics.accuracy_score(y_test,y_pred)

print(f"Accuracy: {acc}")
print(classification_report(y_test,y_pred)) 


Accuracy: 0.905
              precision    recall  f1-score   support

           0       0.91      0.96      0.93       140
           1       0.89      0.78      0.83        60

    accuracy                           0.91       200
   macro avg       0.90      0.87      0.88       200
weighted avg       0.90      0.91      0.90       200



In [5]:
report_dict = classification_report(y_test,y_pred,output_dict=True)
report_dict

{'0': {'precision': 0.9115646258503401,
  'recall': 0.9571428571428572,
  'f1-score': 0.9337979094076655,
  'support': 140.0},
 '1': {'precision': 0.8867924528301887,
  'recall': 0.7833333333333333,
  'f1-score': 0.831858407079646,
  'support': 60.0},
 'accuracy': 0.905,
 'macro avg': {'precision': 0.8991785393402645,
  'recall': 0.8702380952380953,
  'f1-score': 0.8828281582436557,
  'support': 200.0},
 'weighted avg': {'precision': 0.9041329739442946,
  'recall': 0.905,
  'f1-score': 0.9032160587092597,
  'support': 200.0}}

In [ ]:
acc

0.905

In [8]:
mlflow.set_experiment("Classification-Experiment")
mlflow.set_tracking_uri("http://localhost:5000")
with mlflow.start_run(run_name="Logistic Regression"):
    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_params(params)
    mlflow.log_metrics({
        "precision_0": report_dict['0']['precision'],
        "precision_1": report_dict['1']['precision'],
        "recall_score_0": report_dict['0']['recall'],
        "recall_score_1": report_dict['1']['recall'],
        "f1-score_macro": report_dict['macro avg']['f1-score'],
        "f1-score_weighted": report_dict['weighted avg']['f1-score']
        })

    mlflow.sklearn.log_model(lr,"model")



2026/03/03 16:20:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 16:20:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/1/runs/969ca7d81ad842d88307ef11d82bf193
🧪 View experiment at: http://localhost:5000/#/experiments/1


# Dealing with Multiple model and experimenting
1. Random Forest
2. Logistic Regression
3. XgBoost

In [6]:
import sklearn
import pandas as pd
import numpy as np
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report
from sklearn.model_selection import train_test_split
from mlflow import log_metric, log_param, log_artifact
from mlflow.sklearn import log_model
from sklearn.datasets import make_classification    
from xgboost.sklearn import XGBClassifier

# Load the dataset

In [7]:
X,y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42,weights=[0.7,0.3])
np.unique(y,return_counts=True) 

(array([0, 1]), array([699, 301]))

In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print(f"Train set: {np.unique(y_train,return_counts=True)}")

Train set: (array([0, 1]), array([559, 241]))


In [12]:
lr = LogisticRegression(C=1, solver="lbfgs")
lr.fit(X_train,y_train)
y_pred_logistic = lr.predict(X_test)
report_logistic = classification_report(y_test,y_pred_logistic,output_dict=True)
print(report_logistic)

{'0': {'precision': 0.9115646258503401, 'recall': 0.9571428571428572, 'f1-score': 0.9337979094076655, 'support': 140.0}, '1': {'precision': 0.8867924528301887, 'recall': 0.7833333333333333, 'f1-score': 0.831858407079646, 'support': 60.0}, 'accuracy': 0.905, 'macro avg': {'precision': 0.8991785393402645, 'recall': 0.8702380952380953, 'f1-score': 0.8828281582436557, 'support': 200.0}, 'weighted avg': {'precision': 0.9041329739442946, 'recall': 0.905, 'f1-score': 0.9032160587092597, 'support': 200.0}}


In [13]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train,y_train)    
y_pred_rf = rf.predict(X_test)
report_rf = classification_report(y_test,y_pred_rf,output_dict=True)

In [14]:
xgb = XGBClassifier(n_estimators=1000, random_state=42,  )
xgb.fit(X_train,y_train)
y_pred_xgb = xgb.predict(X_test)
report_xgb = classification_report(y_test,y_pred_xgb, output_dict=True)

In [9]:
from imblearn.combine import SMOTETomek

smote = SMOTETomek(random_state=42, sampling_strategy='auto')
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

xgboost_smote = XGBClassifier(n_estimators=1000, random_state=42,  )
xgboost_smote.fit(X_train_smote,y_train_smote)
y_pred_xgb_smote = xgboost_smote.predict(X_test)
report_xgb_smote = classification_report(y_test,y_pred_xgb_smote, output_dict=True)
print(report_xgb_smote)

{'0': {'precision': 0.9455782312925171, 'recall': 0.9928571428571429, 'f1-score': 0.9686411149825784, 'support': 140.0}, '1': {'precision': 0.9811320754716981, 'recall': 0.8666666666666667, 'f1-score': 0.9203539823008849, 'support': 60.0}, 'accuracy': 0.955, 'macro avg': {'precision': 0.9633551533821076, 'recall': 0.9297619047619048, 'f1-score': 0.9444975486417317, 'support': 200.0}, 'weighted avg': {'precision': 0.9562443845462713, 'recall': 0.955, 'f1-score': 0.9541549751780704, 'support': 200.0}}


## Tracking experimens with MLFLOW

In [10]:

models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        { "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_smote, y_train_smote),
        (X_test, y_test)
    )
]

In [23]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

c:\Users\Sainath\anaconda3\envs\mlflow_env\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:32:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [35]:
reports

[{'0': {'precision': 0.9115646258503401,
   'recall': 0.9571428571428572,
   'f1-score': 0.9337979094076655,
   'support': 140.0},
  '1': {'precision': 0.8867924528301887,
   'recall': 0.7833333333333333,
   'f1-score': 0.831858407079646,
   'support': 60.0},
  'accuracy': 0.905,
  'macro avg': {'precision': 0.8991785393402645,
   'recall': 0.8702380952380953,
   'f1-score': 0.8828281582436557,
   'support': 200.0},
  'weighted avg': {'precision': 0.9041329739442946,
   'recall': 0.905,
   'f1-score': 0.9032160587092597,
   'support': 200.0}},
 {'0': {'precision': 0.8947368421052632,
   'recall': 0.9714285714285714,
   'f1-score': 0.9315068493150684,
   'support': 140.0},
  '1': {'precision': 0.9166666666666666,
   'recall': 0.7333333333333333,
   'f1-score': 0.8148148148148148,
   'support': 60.0},
  'accuracy': 0.9,
  'macro avg': {'precision': 0.9057017543859649,
   'recall': 0.8523809523809524,
   'f1-score': 0.8731608320649416,
   'support': 200.0},
  'weighted avg': {'precision':

In [11]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [24]:
# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name= model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, name = model_name)
        else:
            mlflow.sklearn.log_model(model,name = model_name) 

2026/03/03 17:32:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/3/runs/842c372267e94a5eb2b64de8a33f7b51
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/03/03 17:32:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: http://localhost:5000/#/experiments/3/runs/7379c3a013424aec8ecd6d27ac45baef
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/3/runs/ee8360cd14724fcd9a5aee96199a4508
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/3/runs/660f811915ad4a6cbe5f8533de163088
🧪 View experiment at: http://localhost:5000/#/experiments/3


# Using Auto  Logger

In [20]:
# turn on automatic logging for the libraries we use
mlflow.sklearn.autolog()
mlflow.xgboost.autolog()

# make sure the experiment/URI are set (can be the same as before or a new one)
mlflow.set_experiment("Anomaly Detection Auto")
mlflow.set_tracking_uri("http://localhost:5000")

for model_name, params, model, train_set, test_set in models:
    X_tr, y_tr = train_set
    X_te, y_te = test_set

    with mlflow.start_run(run_name=model_name):
        model.set_params(**params)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        acc = accuracy_score(y_te, y_pred)
        print(f"{model_name} accuracy: {acc:.4f}")
        # autolog has already recorded params/metrics/model;
        # we can still log any extra metric if desired
        mlflow.log_metric("test_accuracy", acc)

2026/03/03 16:25:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic Regression accuracy: 0.9050
🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/4/runs/322d26a1d49e449282bcc360c2ca7016
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/03/03 16:25:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest accuracy: 0.9100
🏃 View run Random Forest at: http://localhost:5000/#/experiments/4/runs/0999711f0f074e569f577be7316e116c
🧪 View experiment at: http://localhost:5000/#/experiments/4


c:\Users\Sainath\anaconda3\envs\mlflow_env\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:26:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/03/03 16:26:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBClassifier accuracy: 0.9500
🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/4/runs/1f7cc904427e464d8bae99a95166ee7c
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/03/03 16:26:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBClassifier With SMOTE accuracy: 0.9500
🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/4/runs/3da11ca246e545388706a4730a96fe4a
🧪 View experiment at: http://localhost:5000/#/experiments/4


## Register the model       

In [13]:
model_name = "XGBClassifier With SMOTE"


In [31]:
#XGB_SMOTE = "05d9e7600cbc428495e0ddba4e662b37"
model_name = "XGBClassifier With SMOTE"
run_id =input("Enter Model Run Id")
model_uri = f"runs:/{run_id}/{model_name}"

result = mlflow.register_model(model_uri, model_name)




Registered model 'XGBClassifier With SMOTE' already exists. Creating a new version of this model...
2026/03/03 17:42:26 WARNING mlflow.tracking._model_registry.fluent: Run with id 660f811915ad4a6cbe5f8533de163088 has no artifacts at artifact path 'XGBClassifier With SMOTE', registering model based on models:/m-3c8dfc5a7cae469992a940be477ad7a7 instead
2026/03/03 17:42:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBClassifier With SMOTE, version 4
Created version '4' of model 'XGBClassifier With SMOTE'.


# Load the registered model

In [32]:
model_uri = f"models:/{model_name}@challenger_4"
loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
y_pred[:10]

array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0])

In [34]:
dev_model_uri = f"models:/{model_name}@challenger_4"

prod_model_uri = f"anaomly_detection"

client= mlflow.MlflowClient()
client.copy_model_version(dev_model_uri, prod_model_uri)


Successfully registered model 'anaomly_detection'.
Copied version '4' of model 'XGBClassifier With SMOTE' to version '1' of model 'anaomly_detection'.


<ModelVersion: aliases=[], creation_timestamp=1772540076781, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1772540076781, metrics=None, model_id=None, name='anaomly_detection', params=None, run_id='660f811915ad4a6cbe5f8533de163088', run_link='', source='models:/XGBClassifier With SMOTE/4', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>